***

Preparing Workspace

***

In [ ]:



EXPORT=False


import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from IPython.display import display
pd.set_option('display.max_columns', None)


PATH_GIT = Path.cwd().parent.parent
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_GIT / 'Data' / 'FFIEC' / 'config'

# SharePoint OneDrive paths
PATH_MAIN = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_ORIG = Path(r'I:\Projects\Josh\Regional Monitoring\Task 9. Collect new data\FFIEC')

import sys
sys.path.append(str(PATH_CONFIG0))
import functions as func
import plot as pt

sys.path.append(str(PATH_CONFIG))
import ffiec




In [ ]:

# Importing

years    = ['2018', '2019', '2020', '2021', '2022', '2023']
counties = ['06101','06115','06113','06061','06017','06067']

dtypes = {
    'loan_to_value_ratio'           : str
    , 'interest_rate'               : str
    , 'rate_spread'                 : str
    , 'total_loan_costs'            : str
    , 'total_points_and_fees'       : str
    , 'origination_charges'         : str
    , 'discount_points'             : str
    , 'lender_credits'              : str
    , 'loan_term'                   : str
    , 'prepayment_penalty_term'     : str
    , 'intro_rate_period'           : str
    , 'property_value'              : str
    , 'total_units'                 : str
    , 'multifamily_affordable_units': str
}


df_hmda_new = ffiec.hmda_import(years, counties, dtypes)
df_hmda_new = ffiec.hmda_process(df_hmda_new)


counties = ['Sacramento County', 'Placer County', 'El Dorado County', 'Yuba County', 'Yolo County', 'Sutter County']

dtypes = {
    'applicant_race_name_3'     : str
    , 'applicant_race_name_4'   : str
    , 'applicant_race_name_5'   : str
    , 'co_applicant_race_name_2': str
    , 'co_applicant_race_name_3': str
    , 'co_applicant_race_name_4': str
    , 'co_applicant_race_name_5': str
    , 'denial_reason_name_1'    : str
    , 'denial_reason_name_2'    : str
    , 'denial_reason_name_3'    : str
}


df_hmda_vintage = ffiec.hmda_import_vintage(PATH_ORIG / '2007-2017', counties, dtypes)
df_hmda_vintage = ffiec.hmda_process_vintage(df_hmda_vintage)


print(''); print(''); print('All years together: ')
df_hmda = pd.concat([df_hmda_new, df_hmda_vintage])
df_hmda = df_hmda.reset_index(drop=True)
df_hmda.head()



In [ ]:


## Get overall SACOG estimates
df_hmda2 = df_hmda.copy()


df_hmda2 = df_hmda2[['year', 'county_name', 'purpose', 'demographic', 'origination_rate', 'total']]
wm = lambda x: np.average(x, weights = df_hmda2.loc[x.index, "total"]) # weighted average

df_mpo = df_hmda2.groupby(['year', 'purpose', 'demographic'], as_index=False, sort=False).agg(
    total = ('total', 'sum')
    , origination_rate = ('origination_rate', wm)
)
df_mpo['county_name'] = 'SACOG'

df_mpo = df_mpo.set_index(['county_name', 'year', 'purpose', 'demographic']).reset_index()

df_mpo['Sort'] = pd.Categorical(df_mpo['demographic'], [
    'Asian'
    , 'Black or African American'
    , 'Hispanic or Latino'
    , 'White'
    , 'Not Hispanic or Latino'
    , 'Non-White'
    , 'Male'
    , 'Female'
])

df_mpo = df_mpo.sort_values(['year', 'purpose', 'Sort'], ascending = [False, True, True])
df_mpo = df_mpo.drop('Sort', axis=1)
display(df_mpo.head())


df_hmda3 = pd.concat([df_hmda2, df_mpo])
df_mpo = df_mpo.rename(columns = {'county_name':'MPO'})
display(df_hmda3.head())




In [ ]:


## Loan origination rates
df_rate = df_hmda3.copy()
df_rate = df_rate[['year', 'county_name', 'purpose', 'demographic', 'origination_rate']]


## Loan origination rate gap
df_gap = df_hmda3.copy()

df_gap = df_gap[['year', 'county_name', 'purpose', 'demographic', 'origination_rate']]
df_gap = df_gap.pivot_table(index = ['year', 'county_name', 'purpose']
                                            , columns = 'demographic'
                                            , values = 'origination_rate').reset_index()

df_gap['Female to Male'                              ] = df_gap['Female'                   ] - df_gap['Male'                  ]
df_gap['Hispanic or Latino to Not Hispanic or Latino'] = df_gap['Hispanic or Latino'       ] - df_gap['Not Hispanic or Latino']
df_gap['Non-White to White'                          ] = df_gap['Non-White'                ] - df_gap['White'                 ]
df_gap['Asian to White'                              ] = df_gap['Asian'                    ] - df_gap['White'                 ]
df_gap['Black or African American to White'          ] = df_gap['Black or African American'] - df_gap['White'                 ]

df_gap = df_gap[['year', 'county_name', 'purpose',
                 'Female to Male', 'Hispanic or Latino to Not Hispanic or Latino', 
                 'Non-White to White', 'Asian to White', 'Black or African American to White']]

df_gap = pd.melt(df_gap, id_vars = ['year', 'county_name', 'purpose'], var_name = 'demographic', value_name = 'origination_rate')
df_gap['origination_rate'] = df_gap['origination_rate']*100


## Total decisions (originations + denials)
df_rate = df_hmda2.copy()
df_rate = df_rate[['year', 'county_name', 'purpose', 'demographic', 'total']]


## Plotting
df_plot = df_gap.copy()

df_plot['origination_rate'] = round(df_plot['origination_rate'], 1)
df_plot = df_plot[df_plot['purpose'    ] == 'All'  ]
df_plot = df_plot[df_plot['county_name'] == 'SACOG']

color_map = {
    'Female to Male': '#9DC209'
    , 'Hispanic or Latino to Not Hispanic or Latino': '#1E90FF'
    , 'Non-White to White': "#FBB117"
    , 'Asian to White': "#DC381F"
    , 'Black or African American to White': '#1F45FC'
}

fig = px.line(df_plot, x='year', y='origination_rate'
              , color='demographic'
              , color_discrete_map=color_map
              , markers=True)

title = '<b>Mortgage Loan Origination Gap</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_xaxes(dtick=1)
fig.update_traces(hovertemplate="%{y}%")

fig.show()



In [ ]:


indicator_name = 'Burden_1'
year_start = 2007
year_end = 2023
sample_type = 'HMDA'
tag = 'Mortgage Lending'

df_about = func.write_about(sample_type      = sample_type
                            , indicator_name = indicator_name
                            , year_start     = year_start
                            , year_end       = year_end)

display(df_about)



# Set parameters for export file
workbook_name1 = f'{indicator_name} Counties {sample_type}.xlsx'
workbook_name2 = f'{indicator_name} MPO {sample_type}.xlsx'
paths = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development'/'Housing Burden', PATH_SERVER]

for path in paths:

    path = path / f'{indicator_name} {tag}'

    with pd.ExcelWriter(path/workbook_name1, engine='xlsxwriter') as writer:
        df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
        df_hmda3.to_excel(writer, index = False, sheet_name = 'Counties'           )

    with pd.ExcelWriter(path/workbook_name2, engine='xlsxwriter') as writer:
        df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
        df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO'                )

    print('Export Location: ' + str(path))



In [ ]:

## QC ---------------

df_counts = df_hmda[['county_name', 'purpose']].value_counts()
df_counts = pd.DataFrame(df_counts).reset_index()
df_counts = df_counts.sort_values(['county_name', 'purpose'])
# df_counts = df_hmda[['county_name', 'year', 'purpose', 'demographic']].value_counts()

df_sutter = df_hmda[df_hmda['county_name'] == 'Sutter']
df_sutter = df_sutter[['county_name', 'purpose', 'demographic']].value_counts()
df_sutter = pd.DataFrame(df_sutter).reset_index()
df_sutter = df_sutter.sort_values(['county_name', 'purpose', 'demographic'])
# Sutter missing black or african american home improvement record for 2020
# df_sutter = df_hmda[(df_hmda['county_name'] == 'Sutter') & (df_hmda['demographic'] == 'Black or African American')]
